# BYOL Analysis

Detailed analysis of the **BYOL (Bootstrap Your Own Latent)** self-supervised model trained on STL-10 (96x96 images).

Sections:
1. **Training Trajectories** -- loss, k-NN accuracy, and learning rate over epochs
2. **Evaluation Results** -- k-NN sweep and linear probe (full + low-data)
3. **Intermediate Layer Feature Maps** -- visualize what each ResNet-18 layer captures
4. **BYOL-Specific** -- augmented view pairs and online/target encoder similarity

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms, datasets

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from models.byol import BYOL, BYOLAugmentation
from evaluation import extract_features
from evaluation.knn import knn_classify
from evaluation.linear_probe import LinearProbe
from utils.data import get_eval_loaders, STL10_MEAN, STL10_STD, DATA_DIR

sns.set_theme(style="whitegrid", font_scale=1.1)
device = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]
RESULTS_DIR = PROJECT_ROOT / "results" / "byol"

print(f"Device: {device}")
print(f"Results dir: {RESULTS_DIR}")

## Section A: Training Trajectories

Load the training history and plot loss, k-NN accuracy, and learning rate schedule over epochs.

In [ ]:
with open(RESULTS_DIR / "history.json") as f:
    history = json.load(f)

epochs = [entry["epoch"] for entry in history]
losses = [entry["loss"] for entry in history]
knn_accs = [entry["knn_top1"] for entry in history]
lrs = [entry["lr"] for entry in history]

print(f"Training ran for {len(history)} epochs")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Best k-NN accuracy: {max(knn_accs):.4f} (epoch {epochs[np.argmax(knn_accs)]})")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
ax1.plot(epochs, losses, color="steelblue", linewidth=1.5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("BYOL Training Loss")

# k-NN accuracy curve
ax2.plot(epochs, [a * 100 for a in knn_accs], color="darkorange", linewidth=1.5)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("k-NN Top-1 Accuracy (%)")
ax2.set_title("k-NN Accuracy During Training")

# Learning rate schedule
ax3.plot(epochs, lrs, color="seagreen", linewidth=1.5)
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Learning Rate")
ax3.set_title("Learning Rate Schedule")

plt.tight_layout()
plt.show()

## Section B: Evaluation Results

Load the best checkpoint, extract features from the online encoder, then evaluate with k-NN sweep and linear probe.

In [ ]:
# Load pre-computed evaluation results across all checkpoints
with open(PROJECT_ROOT / "results" / "all_eval_results.json") as f:
    all_results = json.load(f)

byol_results = sorted(
    [r for r in all_results if r["method"] == "byol" and r["checkpoint_name"].startswith("checkpoint_")],
    key=lambda r: r["epoch"],
)

eval_epochs = [r["epoch"] for r in byol_results]
eval_knn20 = [r["knn"]["20"] for r in byol_results]
eval_probe = [r["linear_probe"] for r in byol_results]
eval_probe_low = [r["linear_probe_lowdata"] for r in byol_results]

print(f"Loaded {len(byol_results)} checkpoint evaluations")

In [ ]:
# Evaluation metrics across training
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(eval_epochs, [a * 100 for a in eval_knn20], "o-", color="steelblue")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("k-NN Accuracy (k=20)")

axes[1].plot(eval_epochs, [a * 100 for a in eval_probe], "o-", color="darkorange")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Linear Probe (full data)")

axes[2].plot(eval_epochs, [a * 100 for a in eval_probe_low], "o-", color="seagreen")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Accuracy (%)")
axes[2].set_title("Linear Probe (1% data)")

fig.suptitle("BYOL — Evaluation Across Training", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# k-NN accuracy vs k for the best checkpoint (by linear probe)
best_idx = int(np.argmax(eval_probe))
best_result = byol_results[best_idx]
best_knn = best_result["knn"]

fig, ax = plt.subplots(figsize=(8, 5))
k_vals = [int(k) for k in best_knn.keys()]
accs = [best_knn[str(k)] * 100 for k in k_vals]
ax.plot(k_vals, accs, "o-", color="steelblue")
ax.set_xscale("log"); ax.set_xlabel("k"); ax.set_ylabel("Accuracy (%)")
ax.set_title(f"k-NN Accuracy vs k (epoch {best_result['epoch']})")
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
print(f"{'Checkpoint':<25s} {'Epoch':>6s} {'k-NN(20)':>10s} {'Probe':>10s} {'Probe(1%)':>10s}")
print("-" * 63)
for r in byol_results:
    marker = " <-- best" if r is best_result else ""
    print(f"{r['checkpoint_name']:<25s} {r['epoch']:>6d} "
          f"{r['knn']['20']*100:>9.2f}% {r['linear_probe']*100:>9.2f}% "
          f"{r['linear_probe_lowdata']*100:>9.2f}%{marker}")

## Section C: Intermediate Layer Feature Maps

Hook into ResNet-18 layers to visualize what each stage of the online encoder has learned. We show the first 16 channels from each of `layer1` (64ch), `layer2` (128ch), `layer3` (256ch), and `layer4` (512ch).

In [ ]:
# Load model (best by linear probe) and data for visualization sections
best_ckpt_path = RESULTS_DIR / best_result["checkpoint_name"]
ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=True)
ckpt_args = ckpt["args"]
model = BYOL(
    backbone=ckpt_args.get("backbone", "resnet18"),
    proj_output_dim=ckpt_args.get("proj_dim", 256),
    momentum=ckpt_args.get("momentum", 0.996),
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()

loaders = get_eval_loaders(batch_size=256)
print(f"Loaded BYOL from epoch {best_result['epoch']}")

In [ ]:
# Register forward hooks on each ResNet-18 layer
activations = {}

def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

hooks = []
for name in ["layer1", "layer2", "layer3", "layer4"]:
    h = getattr(model.encoder, name).register_forward_hook(make_hook(name))
    hooks.append(h)

# Get a single test image
test_ds = datasets.STL10(root=str(DATA_DIR), split="test", download=False,
                         transform=transforms.Compose([
                             transforms.Resize(96),
                             transforms.CenterCrop(96),
                             transforms.ToTensor(),
                             transforms.Normalize(mean=STL10_MEAN, std=STL10_STD),
                         ]))
test_img, test_label = test_ds[0]
print(f"Test image class: {CLASS_NAMES[test_label]}")
print(f"Test image shape: {tuple(test_img.shape)}")

# Forward pass to capture activations
with torch.no_grad():
    _ = model.encoder(test_img.unsqueeze(0).to(device))

# Remove hooks
for h in hooks:
    h.remove()

for name in ["layer1", "layer2", "layer3", "layer4"]:
    print(f"{name}: {tuple(activations[name].shape)}")

In [ ]:
def denormalize(tensor, mean=STL10_MEAN, std=STL10_STD):
    t = tensor.clone()
    for i, (m, s) in enumerate(zip(mean, std)):
        t[i].mul_(s).add_(m)
    return t.clamp(0, 1)

layer_names = ["layer1", "layer2", "layer3", "layer4"]

# Show original image
fig_orig, ax_orig = plt.subplots(figsize=(3, 3))
orig_img = denormalize(test_img).permute(1, 2, 0).numpy()
ax_orig.imshow(orig_img)
ax_orig.set_title(f"Original -- {CLASS_NAMES[test_label]}", fontsize=12, fontweight="bold")
ax_orig.axis("off")
plt.tight_layout()
plt.show()

# For each layer, show first 16 channels as a 4x4 grid
for layer_name in layer_names:
    act = activations[layer_name][0]  # (C, H, W)
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    fig.suptitle(f"{layer_name} -- first 16 of {act.shape[0]} channels ({act.shape[1]}x{act.shape[2]})",
                 fontsize=13, fontweight="bold")
    for i in range(16):
        row, col = divmod(i, 4)
        axes[row, col].imshow(act[i].numpy(), cmap="viridis")
        axes[row, col].set_title(f"Ch {i}", fontsize=9)
        axes[row, col].axis("off")
    plt.tight_layout()
    plt.show()

## Section D: BYOL-Specific -- Augmented Views + Online/Target Similarity

### Augmented View Pairs

BYOL learns by predicting target features from augmented views. Here we visualize the pairs of augmented views that the model trains on, generated by `BYOLAugmentation()`.

In [ ]:
# Load raw PIL images (no transform) to show original + augmented views
raw_ds = datasets.STL10(root=str(DATA_DIR), split="test", download=False)
aug = BYOLAugmentation()

fig, axes = plt.subplots(4, 3, figsize=(10, 13))
axes[0, 0].set_title("Original", fontsize=12, fontweight="bold")
axes[0, 1].set_title("View 1", fontsize=12, fontweight="bold")
axes[0, 2].set_title("View 2", fontsize=12, fontweight="bold")

for i in range(4):
    pil_img, label = raw_ds[i]
    view1, view2 = aug(pil_img)

    # Original
    axes[i, 0].imshow(pil_img)
    axes[i, 0].set_ylabel(CLASS_NAMES[label], fontsize=11, fontweight="bold")
    axes[i, 0].set_xticks([])
    axes[i, 0].set_yticks([])

    # View 1 (denormalize)
    v1_display = denormalize(view1).permute(1, 2, 0).numpy()
    axes[i, 1].imshow(v1_display)
    axes[i, 1].axis("off")

    # View 2 (denormalize)
    v2_display = denormalize(view2).permute(1, 2, 0).numpy()
    axes[i, 2].imshow(v2_display)
    axes[i, 2].axis("off")

plt.suptitle("BYOL Augmented View Pairs", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Online vs Target Encoder Similarity

BYOL maintains two encoders: an **online** encoder (updated by gradient descent) and a **target** encoder (updated via exponential moving average). After training, the two should produce highly similar representations for the same input.

We compute per-sample cosine similarities between online and target features, and visualize the distribution and cross-sample similarity matrix.

In [ ]:
# Get a batch of test images
test_loader_iter = iter(loaders["test"])
batch, batch_labels = next(test_loader_iter)

# Compute features from both encoders
with torch.no_grad():
    online_feats = model.encoder(batch.to(device))
    target_feats = model.target_encoder(batch.to(device))

online_norm = F.normalize(online_feats, dim=1)
target_norm = F.normalize(target_feats, dim=1)

# Per-sample cosine similarity (online vs target for the same image)
cosine_sim = (online_norm * target_norm).sum(dim=1).cpu()

print(f"Batch size: {batch.shape[0]}")
print(f"Online features: {tuple(online_feats.shape)}")
print(f"Target features: {tuple(target_feats.shape)}")
print(f"Cosine similarity -- mean: {cosine_sim.mean():.4f}, std: {cosine_sim.std():.4f}")
print(f"Cosine similarity -- min: {cosine_sim.min():.4f}, max: {cosine_sim.max():.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of per-sample cosine similarities
ax1.hist(cosine_sim.numpy(), bins=30, color="steelblue", edgecolor="white", alpha=0.8)
ax1.axvline(cosine_sim.mean().item(), color="red", linestyle="--", linewidth=1.5,
            label=f"Mean = {cosine_sim.mean():.4f}")
ax1.set_xlabel("Cosine Similarity")
ax1.set_ylabel("Count")
ax1.set_title("Per-Sample Online vs Target Cosine Similarity")
ax1.legend()

# Cosine similarity matrix between online and target features
# Use a subset for readability (first 32 samples)
n_show = min(32, batch.shape[0])
sim_matrix = (online_norm[:n_show].cpu() @ target_norm[:n_show].cpu().T).numpy()

im = ax2.imshow(sim_matrix, cmap="viridis", vmin=-1, vmax=1, aspect="equal")
ax2.set_xlabel("Target sample index")
ax2.set_ylabel("Online sample index")
ax2.set_title(f"Online x Target Cosine Similarity Matrix (first {n_show})")
plt.colorbar(im, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

print(f"Diagonal (same-image similarity) mean: {np.diag(sim_matrix).mean():.4f}")
off_diag = sim_matrix[~np.eye(n_show, dtype=bool)]
print(f"Off-diagonal (cross-image similarity) mean: {off_diag.mean():.4f}")